# 🌳 Módulo 3: Técnicas de Classificação em Mineração de Dados

**Curso:** Mineração de Dados | UEPA-CCNT | 8º Semestre

---

## 📋 Objetivos
- Implementar e comparar algoritmos de classificação em dados reais
- Calcular e interpretar métricas de avaliação (acurácia, F1, AUC-ROC, AUC-PR)
- Visualizar fronteiras de decisão em espaço bidimensional (via PCA)
- Aplicar XGBoost com tuning e interpretar com SHAP

## 📑 Conteúdo
1. [Setup e Instalação](#setup)
2. [Dataset Breast Cancer Wisconsin e EDA](#eda)
3. [Árvores de Decisão](#dt)
4. [Random Forest e Feature Importance](#rf)
5. [XGBoost e LightGBM](#xgb)
6. [SVM, KNN e Naive Bayes](#svm-knn-nb)
7. [Avaliação e Comparação Completa](#eval)
8. [Interpretabilidade com SHAP](#shap)
9. [Exercícios Práticos](#exercises)


In [ ]:
# Instalação de dependências necessárias
!pip install -q xgboost lightgbm shap scikit-learn pandas numpy matplotlib seaborn plotly lime

## 📦 1. Importações e Configuração
<a id='setup'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn — datasets, modelos e métricas
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# Classificadores
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, roc_curve, precision_recall_curve
)

# XGBoost e LightGBM
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# SHAP para interpretabilidade
import shap

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Configuração visual
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3
})
sns.set_theme(style='whitegrid', palette='muted')

print('Bibliotecas carregadas com sucesso!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

## 🗂️ 2. Dataset: Breast Cancer Wisconsin
<a id='eda'></a>

O **Breast Cancer Wisconsin** contém 569 amostras de tumores com 30 features numéricas extraídas de imagens de citologia de aspiração de agulha fina (FNA). O objetivo é classificar tumores como **maligno (1)** ou **benigno (0)**.

In [ ]:
# ── Carregamento e preparação do dataset ──────────────────────────────────────
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['target_name'] = df['target'].map({0: 'Maligno', 1: 'Benigno'})

print('=' * 55)
print(f'  Breast Cancer Wisconsin — Resumo')
print('=' * 55)
print(f'  Amostras:  {df.shape[0]}')
print(f'  Features:  {df.shape[1] - 2}')
print(f'  Classes:   Maligno (0) e Benigno (1)')
print()

# Distribuição de classes
class_counts = df['target_name'].value_counts()
print('Distribuição de Classes:')
for cls, cnt in class_counts.items():
    pct = cnt / len(df) * 100
    print(f'  {cls}: {cnt} ({pct:.1f}%)')

print()
print('Primeiras 5 linhas:')
df[cancer.feature_names[:6]].head()

In [ ]:
# ── EDA Visual: Correlação e Violino ──────────────────────────────────────────
features = cancer.feature_names
X = df[features]
y = df['target']

# Correlação com o target
correlations = X.corrwith(y).abs().sort_values(ascending=False)
top10 = correlations.head(10).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap de correlação (top 10 features)
corr_matrix = df[top10 + ['target']].corr()
mask = np.zeros_like(corr_matrix)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=axes[0],
    annot_kws={'size': 7}, linewidths=0.5
)
axes[0].set_title('Correlação — Top 10 Features vs. Target', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)

# Violin plots para as top 4 features mais correlacionadas
top4 = top10[:4]
df_melt = df[top4 + ['target_name']].melt(
    id_vars='target_name', var_name='Feature', value_name='Valor'
)
sns.violinplot(
    data=df_melt, x='Feature', y='Valor', hue='target_name',
    ax=axes[1], split=False, palette=['#e74c3c', '#2ecc71'], inner='quartile'
)
axes[1].set_title('Distribuição das Top 4 Features por Classe', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30, labelsize=8)
axes[1].legend(title='Classe')

plt.suptitle('Breast Cancer Wisconsin — Análise Exploratória', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Divisão Treino / Teste ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Normalização (necessário para SVM, KNN, LR)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Divisão Treino/Teste (stratified):')
print(f'  X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'  X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print()
print('Proporção de classes no treino:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Proporção de classes no teste: ', y_test.value_counts(normalize=True).round(3).to_dict())

## 🌳 3. Árvores de Decisão
<a id='dt'></a>

Exploraremos o efeito da profundidade máxima (`max_depth`) no desempenho e visualizaremos a árvore treinada.

In [ ]:
# ── Efeito da Profundidade no Desempenho ─────────────────────────────────────
depths = [2, 3, 4, 5, 6, 7, None]
train_accs, test_accs, f1_scores = [], [], []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test, dt.predict(X_test)))
    f1_scores.append(f1_score(y_test, dt.predict(X_test)))

depth_labels = [str(d) if d else 'None\n(sem limite)' for d in depths]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Curva de acurácia vs profundidade
axes[0].plot(range(len(depths)), train_accs, 'o-', color='#e74c3c', label='Treino', lw=2)
axes[0].plot(range(len(depths)), test_accs, 's-', color='#2980b9', label='Teste', lw=2)
axes[0].set_xticks(range(len(depths)))
axes[0].set_xticklabels(depth_labels, fontsize=8)
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Acurácia')
axes[0].set_title('Acurácia vs. Profundidade da Árvore', fontweight='bold')
axes[0].legend()
axes[0].set_ylim([0.85, 1.01])

# F1-Score vs profundidade
axes[1].bar(range(len(depths)), f1_scores, color='#27ae60', edgecolor='white', alpha=0.85)
axes[1].set_xticks(range(len(depths)))
axes[1].set_xticklabels(depth_labels, fontsize=8)
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1-Score vs. Profundidade da Árvore', fontweight='bold')
axes[1].set_ylim([0.90, 1.01])
for i, v in enumerate(f1_scores):
    axes[1].text(i, v + 0.001, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Decisão Tree — Efeito da Profundidade Máxima', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Melhor profundidade pelo teste
best_idx = np.argmax(test_accs)
print(f'Melhor max_depth pelo teste: {depths[best_idx]} (acurácia={test_accs[best_idx]:.4f})')

In [ ]:
# ── Visualização da Árvore e Feature Importances ─────────────────────────────
dt_best = DecisionTreeClassifier(max_depth=4, random_state=SEED)
dt_best.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Visualização da árvore
plot_tree(
    dt_best, feature_names=list(features), class_names=['Maligno', 'Benigno'],
    filled=True, rounded=True, max_depth=3, fontsize=6,
    impurity=True, proportion=False, ax=axes[0]
)
axes[0].set_title('Árvore de Decisão (max_depth=4, exibida até 3)', fontweight='bold')

# Feature importances
fi = pd.Series(dt_best.feature_importances_, index=features).sort_values(ascending=True).tail(15)
fi.plot(kind='barh', ax=axes[1], color='#2980b9', edgecolor='white', alpha=0.85)
axes[1].set_title('Top 15 Feature Importances (Impurity, max_depth=4)', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

# ── Cálculo manual de Entropia e Gini ────────────────────────────────────────
print('\n── Cálculo Manual de Entropia e Gini ────────────────────────────')
def entropia(p_positivo):
    if p_positivo in [0, 1]:
        return 0.0
    p_neg = 1 - p_positivo
    return -p_positivo * np.log2(p_positivo) - p_neg * np.log2(p_neg)

def gini(p_positivo):
    return 1 - p_positivo**2 - (1 - p_positivo)**2

# Exemplo com proporção de benignos no conjunto de treino
p_benigno = y_train.mean()
print(f'Proporção de Benignos no treino: {p_benigno:.4f}')
print(f'Entropia H(S) = {entropia(p_benigno):.4f} bits')
print(f'Gini(S)       = {gini(p_benigno):.4f}')
print(f'(Entropia máxima com 2 classes: 1.0 bit | Gini máx: 0.5)')

## 🌲 4. Random Forest e Feature Importance
<a id='rf'></a>

In [ ]:
# ── Random Forest com GridSearchCV ───────────────────────────────────────────
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(random_state=SEED, oob_score=True, n_jobs=-1)
grid_rf = GridSearchCV(
    rf_base, param_grid_rf, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='f1', n_jobs=-1, verbose=0
)
grid_rf.fit(X_train, y_train)

rf_best = grid_rf.best_estimator_
print('Melhores hiperparâmetros (GridSearchCV):')
for k, v in grid_rf.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nOOB Score: {rf_best.oob_score_:.4f}')
print(f'F1 (Teste): {f1_score(y_test, rf_best.predict(X_test)):.4f}')
print(f'AUC-ROC (Teste): {roc_auc_score(y_test, rf_best.predict_proba(X_test)[:,1]):.4f}')

# Feature importances: RF vs DT
fi_rf = pd.Series(rf_best.feature_importances_, index=features)
fi_dt = pd.Series(dt_best.feature_importances_, index=features)
top_features = fi_rf.sort_values(ascending=False).head(12).index

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fi_vals, title, color in zip(
    axes,
    [fi_rf[top_features].sort_values(), fi_dt[top_features].sort_values()],
    ['Random Forest — Feature Importance (MDI)', 'Decision Tree — Feature Importance (MDI)'],
    ['#2980b9', '#e74c3c']
):
    fi_vals.plot(kind='barh', ax=ax, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importância (Gini)')

plt.suptitle('Importância de Features: Random Forest vs. Decision Tree', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Permutation Importance vs Impurity Importance ────────────────────────────
perm_imp = permutation_importance(
    rf_best, X_test, y_test,
    n_repeats=20, random_state=SEED, scoring='f1'
)
fi_perm = pd.Series(perm_imp.importances_mean, index=features)

top10_rf  = fi_rf.nlargest(10).index
comparison = pd.DataFrame({
    'MDI (Impurity)': fi_rf[top10_rf],
    'Permutation':    fi_perm[top10_rf]
})

ax = comparison.plot(kind='bar', figsize=(12, 4), color=['#2980b9', '#e67e22'], edgecolor='white', alpha=0.85)
plt.title('MDI vs. Permutation Importance — Top 10 Features (Random Forest)', fontweight='bold')
plt.ylabel('Importância')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.legend()
plt.tight_layout()
plt.show()

print('Nota: MDI (Mean Decrease Impurity) pode superestimar features de alta cardinalidade.')
print('Permutation Importance é mais confiável, mas computacionalmente mais custoso.')

## ⚡ 5. XGBoost e LightGBM
<a id='xgb'></a>

In [ ]:
# ── XGBoost e LightGBM com Early Stopping ────────────────────────────────────
# Divisão extra para early stopping (validation set)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
    eval_metric='logloss', early_stopping_rounds=30, random_state=SEED,
    verbosity=0
)
xgb_model.fit(
    X_tr2, y_tr2,
    eval_set=[(X_tr2, y_tr2), (X_val, y_val)],
    verbose=False
)

# LightGBM
lgb_model = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    num_leaves=31, random_state=SEED, verbose=-1
)
lgb_model.fit(
    X_tr2, y_tr2,
    eval_set=[(X_val, y_val)],
    callbacks=[]
)

# Comparação de resultados
models_quick = {
    'Random Forest': rf_best,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model
}

print(f'{'Modelo':<20} {'Acurácia':>10} {'F1':>10} {'AUC-ROC':>10}')
print('-' * 52)
for name, mdl in models_quick.items():
    preds = mdl.predict(X_test)
    proba = mdl.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    print(f'{name:<20} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}')

# Curva de aprendizado do XGBoost
results_xgb = xgb_model.evals_result()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_xgb['validation_0']['logloss'], label='Treino', color='#e74c3c', lw=1.5)
ax.plot(results_xgb['validation_1']['logloss'], label='Validação', color='#2980b9', lw=1.5)
ax.axvline(x=xgb_model.best_iteration, color='green', linestyle='--', lw=1, label=f'Best iter={xgb_model.best_iteration}')
ax.set_xlabel('Número de Árvores')
ax.set_ylabel('Log Loss')
ax.set_title('XGBoost — Curva de Aprendizado (Log Loss)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── XGBoost Hyperparameter Tuning com RandomizedSearchCV ─────────────────────
from scipy.stats import randint, uniform

param_dist_xgb = {
    'n_estimators': randint(100, 400),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.29),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0.5, 2)
}

xgb_tune = XGBClassifier(
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, verbosity=0
)

random_search_xgb = RandomizedSearchCV(
    xgb_tune, param_dist_xgb, n_iter=30,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc', n_jobs=-1, random_state=SEED, verbose=0
)
random_search_xgb.fit(X_train, y_train)

xgb_tuned = random_search_xgb.best_estimator_
print('Melhores Hiperparâmetros (XGBoost — RandomizedSearchCV):')
for k, v in random_search_xgb.best_params_.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')
print(f'\nAUC-ROC CV: {random_search_xgb.best_score_:.4f}')
print(f'AUC-ROC Teste: {roc_auc_score(y_test, xgb_tuned.predict_proba(X_test)[:,1]):.4f}')
print(f'F1 Teste: {f1_score(y_test, xgb_tuned.predict(X_test)):.4f}')

## 🔀 6. SVM, KNN e Naive Bayes
<a id='svm-knn-nb'></a>

In [ ]:
# ── SVM, KNN e Naive Bayes — Cross-Validation ─────────────────────────────────
import time

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

classifiers = {
    'SVM Linear':     SVC(kernel='linear', C=1.0, probability=True, random_state=SEED),
    'SVM RBF':        SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=SEED),
    'SVM Poly(d=3)':  SVC(kernel='poly', degree=3, C=1.0, probability=True, random_state=SEED),
    'KNN k=5':        KNeighborsClassifier(n_neighbors=5),
    'KNN k=11':       KNeighborsClassifier(n_neighbors=11),
    'GaussianNB':     GaussianNB()
}

results_svm_knn = {}
for name, clf in classifiers.items():
    # SVMs precisam de dados normalizados
    if 'SVM' in name or 'KNN' in name:
        Xtr, Xte = X_train_sc, X_test_sc
    else:
        Xtr, Xte = X_train.values, X_test.values

    t0 = time.time()
    clf.fit(Xtr, y_train)
    tempo = time.time() - t0

    preds = clf.predict(Xte)
    proba = clf.predict_proba(Xte)[:, 1]

    results_svm_knn[name] = {
        'Acurácia': accuracy_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'AUC-ROC': roc_auc_score(y_test, proba),
        'Tempo(s)': tempo
    }

df_svm_knn = pd.DataFrame(results_svm_knn).T.sort_values('AUC-ROC', ascending=False)
print('SVM, KNN e Naive Bayes — Desempenho no Conjunto de Teste:')
print(df_svm_knn.round(4).to_string())

In [ ]:
# ── Fronteiras de Decisão (PCA 2D) ───────────────────────────────────────────
# Redução para 2D via PCA para visualização
pca = PCA(n_components=2, random_state=SEED)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d  = pca.transform(X_test_sc)

clfs_2d = {
    'SVM RBF':   SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED),
    'SVM Linear': SVC(kernel='linear', C=1, probability=True, random_state=SEED),
    'KNN k=5':   KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'DT (d=4)':  DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'Log. Reg.': LogisticRegression(max_iter=1000, random_state=SEED)
}

# Grade para meshgrid
h = 0.05
x_min, x_max = X_train_2d[:, 0].min() - 0.5, X_train_2d[:, 0].max() + 0.5
y_min, y_max = X_train_2d[:, 1].min() - 0.5, X_train_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
colors_map = plt.cm.RdYlBu

for ax, (name, clf) in zip(axes.ravel(), clfs_2d.items()):
    clf.fit(X_train_2d, y_train)
    if hasattr(clf, 'predict_proba'):
        Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    else:
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.7, cmap=colors_map)
    scatter = ax.scatter(
        X_test_2d[:, 0], X_test_2d[:, 1], c=y_test,
        cmap=colors_map, edgecolors='k', linewidths=0.5, s=25
    )
    test_acc = accuracy_score(y_test, clf.predict(X_test_2d))
    ax.set_title(f'{name}\nAcc={test_acc:.3f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('PC1', fontsize=8)
    ax.set_ylabel('PC2', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('Fronteiras de Decisão — Espaço PCA 2D (Teste)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 📊 7. Avaliação e Comparação Completa
<a id='eval'></a>

In [ ]:
# ── Comparação Completa — Todos os Classificadores ───────────────────────────
import time

all_classifiers = {
    'Logistic Reg.': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))]),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=None, random_state=SEED, n_jobs=-1, oob_score=True),
    'XGBoost':       XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, use_label_encoder=False, eval_metric='logloss', random_state=SEED, verbosity=0),
    'LightGBM':      LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=SEED, verbose=-1),
    'SVM RBF':       Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED))]),
    'KNN k=7':       Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=7))]),
    'Naive Bayes':   GaussianNB()
}

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
comparison_results = {}

for name, clf in all_classifiers.items():
    t0 = time.time()
    acc_cv  = cross_val_score(clf, X_train, y_train, cv=cv10, scoring='accuracy', n_jobs=-1)
    f1_cv   = cross_val_score(clf, X_train, y_train, cv=cv10, scoring='f1', n_jobs=-1)
    auc_cv  = cross_val_score(clf, X_train, y_train, cv=cv10, scoring='roc_auc', n_jobs=-1)
    tempo   = time.time() - t0

    # Treinar no conjunto completo de treino e avaliar no teste
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]

    comparison_results[name] = {
        'Acc CV (mean)':  acc_cv.mean(),
        'Acc CV (std)':   acc_cv.std(),
        'F1 CV (mean)':   f1_cv.mean(),
        'F1 CV (std)':    f1_cv.std(),
        'AUC CV (mean)':  auc_cv.mean(),
        'AUC CV (std)':   auc_cv.std(),
        'F1 Teste':       f1_score(y_test, preds),
        'AUC Teste':      roc_auc_score(y_test, proba),
        'MCC':            matthews_corrcoef(y_test, preds),
        'Kappa':          cohen_kappa_score(y_test, preds),
        'Tempo CV (s)':   tempo
    }

df_comparison = pd.DataFrame(comparison_results).T.sort_values('AUC CV (mean)', ascending=False)
print('Comparação Completa — 10-Fold Stratified CV + Teste:')
print(df_comparison[['Acc CV (mean)', 'F1 CV (mean)', 'AUC CV (mean)', 'F1 Teste', 'AUC Teste', 'MCC']].round(4).to_string())

# Salvar para uso posterior
all_clfs_fitted = {name: clf for name, clf in all_classifiers.items()}

In [ ]:
# ── Curvas ROC e Precision-Recall ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(all_classifiers)))

for (name, clf), color in zip(all_classifiers.items(), colors):
    proba = clf.predict_proba(X_test)[:, 1]

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, lw=1.8, color=color, label=f'{name} (AUC={auc_val:.3f})')

    # PR Curve
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, lw=1.8, color=color, label=f'{name} (AP={ap:.3f})')

# ROC: baseline
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatório (AUC=0.5)')
axes[0].set_xlabel('Taxa de Falsos Positivos')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall)')
axes[0].set_title('Curvas ROC — Todos os Classificadores', fontweight='bold')
axes[0].legend(fontsize=7, loc='lower right')

# PR: baseline
baseline_pr = y_test.mean()
axes[1].axhline(y=baseline_pr, color='k', linestyle='--', lw=1, label=f'Aleatório (AP={baseline_pr:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precisão')
axes[1].set_title('Curvas Precisão-Recall — Todos os Classificadores', fontweight='bold')
axes[1].legend(fontsize=7, loc='lower left')

plt.suptitle('Análise de Desempenho — Curvas de Avaliação', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrizes de Confusão — Top 3 Classificadores ─────────────────────────────
top3_names = df_comparison.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name in zip(axes, top3_names):
    clf = all_classifiers[name]
    preds = clf.predict(X_test)
    cm = confusion_matrix(y_test, preds)

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Maligno', 'Benigno'],
        yticklabels=['Maligno', 'Benigno'],
        linewidths=0.5, linecolor='gray',
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    f1_val  = f1_score(y_test, preds)
    acc_val = accuracy_score(y_test, preds)
    ax.set_title(f'{name}\nAcc={acc_val:.3f} | F1={f1_val:.3f}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — Top 3 Classificadores', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔍 8. Interpretabilidade com SHAP
<a id='shap'></a>

SHAP (SHapley Additive exPlanations) usa a teoria dos jogos cooperativos para atribuir a contribuição de cada feature à predição individual.

In [ ]:
# ── SHAP com XGBoost ──────────────────────────────────────────────────────────
# Treinar XGBoost no dataset completo de treino
xgb_shap = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, verbosity=0
)
xgb_shap.fit(X_train, y_train)

# Explainer SHAP do tipo TreeExplainer (mais eficiente para árvores)
explainer = shap.TreeExplainer(xgb_shap)
shap_values = explainer.shap_values(X_test)

print(f'Shape dos SHAP values: {np.array(shap_values).shape}')
print(f'Shape do X_test: {X_test.shape}')
print(f'\nTop 5 Features por |SHAP| médio:')
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top5_shap = pd.Series(mean_abs_shap, index=features).nlargest(5)
for feat, val in top5_shap.items():
    print(f'  {feat}: {val:.4f}')

# SHAP Summary Plot — Beeswarm
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test,
    feature_names=list(features),
    plot_type='dot', max_display=15,
    show=False
)
plt.title('SHAP Summary Plot (Beeswarm) — XGBoost', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# SHAP Summary Plot — Bar
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_values, X_test,
    feature_names=list(features),
    plot_type='bar', max_display=15,
    show=False
)
plt.title('SHAP Feature Importance (|SHAP| médio) — XGBoost', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# SHAP Waterfall — explicação individual
idx_explain = 5  # índice da amostra a explicar
print(f'\nAmostra {idx_explain}: Real={y_test.iloc[idx_explain]}, Predito={xgb_shap.predict(X_test)[idx_explain]}')
explainer_new = shap.Explainer(xgb_shap, X_train)
shap_values_new = explainer_new(X_test)
plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_values_new[idx_explain], max_display=12, show=False)
plt.title(f'SHAP Waterfall — Amostra {idx_explain}', fontweight='bold')
plt.tight_layout()
plt.show()

# SHAP Dependence Plot — feature mais importante
top_feat = list(features)[np.argmax(mean_abs_shap)]
plt.figure(figsize=(8, 4))
shap.dependence_plot(
    top_feat, shap_values, X_test,
    feature_names=list(features),
    show=False
)
plt.title(f'SHAP Dependence Plot — {top_feat}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── LIME — Explicação local com modelo interpretável ─────────────────────────
try:
    from lime import lime_tabular

    lime_explainer = lime_tabular.LimeTabularExplainer(
        X_train.values,
        feature_names=list(features),
        class_names=['Maligno', 'Benigno'],
        mode='classification',
        random_state=SEED
    )

    # Encontrar um exemplo mal classificado pelo melhor modelo
    preds_all = xgb_shap.predict(X_test)
    misclassified = np.where(preds_all != y_test.values)[0]

    if len(misclassified) > 0:
        idx_misc = misclassified[0]
        print(f'Exemplo mal classificado: índice {idx_misc}')
        print(f'  Real: {y_test.iloc[idx_misc]} | Predito: {preds_all[idx_misc]}')

        lime_exp = lime_explainer.explain_instance(
            X_test.values[idx_misc],
            xgb_shap.predict_proba,
            num_features=10
        )
        fig = lime_exp.as_pyplot_figure()
        fig.set_size_inches(9, 5)
        plt.title(f'LIME — Explicação Local (Amostra Mal Classificada {idx_misc})', fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print('Todos os exemplos foram classificados corretamente pelo XGBoost!')
        print('Usando o índice 0 como exemplo de demonstração...')
        lime_exp = lime_explainer.explain_instance(
            X_test.values[0], xgb_shap.predict_proba, num_features=10
        )
        fig = lime_exp.as_pyplot_figure()
        fig.set_size_inches(9, 5)
        plt.tight_layout()
        plt.show()

    print('\nComparação SHAP vs LIME:')
    print('  SHAP: fundamentado em teoria dos jogos; consistente globalmente; mais lento')
    print('  LIME: aproximação local por modelo linear; mais rápido; pode variar com seed')

except ImportError:
    print('LIME não instalado. Execute: !pip install lime')

## 🎓 9. Exercícios Práticos
<a id='exercises'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXERCÍCIO 1: Cálculo manual de Entropia e Ganho de Informação
# ══════════════════════════════════════════════════════════════════════════════

print('EXERCÍCIO 1 — Cálculo de Entropia e Ganho de Informação')
print('=' * 60)

# Dataset de exemplo (binário)
# TODO: substitua pelos dados do exercício do módulo
dados = pd.DataFrame({
    'A1': [1, 1, 0, 0, 1, 0, 1, 0, 1, 0],
    'A2': [0, 1, 0, 1, 1, 0, 0, 1, 1, 0],
    'Classe': [1, 1, 0, 0, 1, 0, 1, 0, 1, 0]
})

def calc_entropia(serie):
    """Calcula a entropia de Shannon de uma série binária."""
    counts = serie.value_counts(normalize=True)
    return -sum(p * np.log2(p) for p in counts if p > 0)

def calc_ig(df, atributo, target='Classe'):
    """Calcula o Ganho de Informação do atributo em relação ao target."""
    H_S = calc_entropia(df[target])
    IG  = H_S
    for val in df[atributo].unique():
        subset = df[df[atributo] == val][target]
        IG -= (len(subset) / len(df)) * calc_entropia(subset)
    return IG

H_total = calc_entropia(dados['Classe'])
IG_A1   = calc_ig(dados, 'A1')
IG_A2   = calc_ig(dados, 'A2')

print(f'H(S) = {H_total:.4f} bits')
print(f'IG(S, A1) = {IG_A1:.4f}')
print(f'IG(S, A2) = {IG_A2:.4f}')
melhor = 'A1' if IG_A1 > IG_A2 else 'A2'
print(f'ID3 escolheria: {melhor} (maior Ganho de Informação)')

# ══════════════════════════════════════════════════════════════════════════════
# EXERCÍCIO 2: Compare AUC-ROC vs AUC-PR em dados desbalanceados
# ══════════════════════════════════════════════════════════════════════════════
print('\nEXERCÍCIO 2 — AUC-ROC vs. AUC-PR em Dados Desbalanceados')
print('=' * 60)

# TODO: Carregue um dataset desbalanceado (ex: detecção de fraude)
# Simulação: criar dataset artificial muito desbalanceado
from sklearn.datasets import make_classification

X_imb, y_imb = make_classification(
    n_samples=1000, n_features=20, n_informative=5,
    weights=[0.97, 0.03], random_state=SEED
)
print(f'Dataset desbalanceado: {y_imb.sum()} positivos de {len(y_imb)} total ({y_imb.mean()*100:.1f}%)')

X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(X_imb, y_imb, test_size=0.3, random_state=SEED, stratify=y_imb)

# Classificador que prediz sempre a classe majoritária
dummy_preds = np.zeros(len(y_imb_te))
dummy_proba = np.zeros(len(y_imb_te))

print(f'Classificador trivial (sempre prediz 0):')
print(f'  Acurácia: {accuracy_score(y_imb_te, dummy_preds):.4f}  ← ENGANOSA!')
print(f'  Precisão: {precision_score(y_imb_te, dummy_preds, zero_division=0):.4f}')
print(f'  Recall:   {recall_score(y_imb_te, dummy_preds, zero_division=0):.4f}')
print(f'  F1:       {f1_score(y_imb_te, dummy_preds, zero_division=0):.4f}')
print(f'  MCC:      {matthews_corrcoef(y_imb_te, dummy_preds):.4f}')
print()
print('Conclusão: para dados desbalanceados, use F1, AUC-PR e MCC!')

# ══════════════════════════════════════════════════════════════════════════════
# EXERCÍCIO 3: Pipeline completo
# ══════════════════════════════════════════════════════════════════════════════
print('\nEXERCÍCIO 3 — Pipeline de Classificação com Nested CV')
print('=' * 60)
print('TODO: Implementar pipeline com:')
print('  1. Pré-processamento (StandardScaler ou MinMaxScaler)')
print('  2. Seleção de features (SelectKBest ou RFE)')
print('  3. Classificador (XGBoost ou Random Forest)')
print('  4. Nested CV: loop externo (avaliação) + loop interno (tuning)')
print('  5. Relatório: AUC-ROC médio ± std, tempo computacional')

# Exemplo de estrutura básica:
from sklearn.feature_selection import SelectKBest, f_classif

pipeline_example = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectKBest(f_classif, k=15)),
    ('clf', XGBClassifier(n_estimators=100, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss'))
])

auc_nested = cross_val_score(pipeline_example, X, y, cv=cv5, scoring='roc_auc', n_jobs=-1)
print(f'\nPipeline básico (SelectKBest k=15 + XGB): AUC = {auc_nested.mean():.4f} ± {auc_nested.std():.4f}')